# MUTCD Multimodal RAG — HPRC (v3)

**Architecture:**
- Outline-driven typed-paragraph chunks (Standard / Guidance / Option / Support) — never strip rule labels.
- BGE-M3 (dense + sparse, hybrid retrieval in one model).
- ColQwen2 multi-vector page retrieval (ColPali family — 2026 SOTA for documents).
- mxbai-rerank-large-v2 cross-encoder reranker.
- NetworkX knowledge graph: Sections / Chunks / Figures / Tables / SignCodes / Categories cross-linked via `cites_*`, `defines`, `depicts`, `mentions`, `illustrated_by`, `kind_of` edges.
- Qdrant local-file store (embedded; no daemon).
- Qwen2.5-VL-7B-Instruct (with 3B fallback) for grounded structured answers.

**Before running**, the ingestion job must have completed at least once:
```bash
cd $SCRATCH/MRAG && sbatch scripts/ingest_v3.slurm
# or, interactively on a login node with WebProxy loaded:
python scripts/ingest_v3.py
```

After ingestion, just run this notebook top-to-bottom. You'll see Markdown answers + figure crops inline.

## 1. Environment sanity check

In [ ]:
import os, sys
from pathlib import Path

SCRATCH = Path(os.environ.get("SCRATCH", "/tmp"))
for k, v in (
    ("HF_HOME",            str(SCRATCH / "hf_cache")),
    ("TRANSFORMERS_CACHE", str(SCRATCH / "hf_cache")),
    ("HF_HUB_CACHE",       str(SCRATCH / "hf_cache" / "hub")),
    ("HF_HUB_DISABLE_TELEMETRY", "1"),
    ("TOKENIZERS_PARALLELISM",   "false"),
):
    os.environ.setdefault(k, v)

# Add the repo to sys.path so `import mrag` works (the repo lives in $SCRATCH/MRAG)
REPO = SCRATCH / "MRAG"
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch
print("python:", sys.version.split()[0])
print("torch :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB)")
print("REPO  :", REPO)

from mrag.config import CFG
print("PDF        :", CFG.pdf_path, "exists?", CFG.pdf_path.exists())
print("chunks     :", CFG.chunks_jsonl, "exists?", CFG.chunks_jsonl.exists())
print("figures    :", CFG.figures_jsonl, "exists?", CFG.figures_jsonl.exists())
print("sign codes :", CFG.sign_codes_json, "exists?", CFG.sign_codes_json.exists())
print("graph      :", CFG.graph_pickle, "exists?", CFG.graph_pickle.exists())
print("qdrant dir :", CFG.qdrant_dir)

## 2. Initialize the pipeline (one time, ~30–60 s)

Loads BGE-M3, ColQwen2, mxbai-rerank-large-v2, the knowledge graph, the Qdrant store, and Qwen2.5-VL-7B (fallback to 3B if it doesn't fit).

If you only want to inspect retrieval (no generation), pass `load_vlm=False` to skip the ~16 GB VLM load.

In [ ]:
from mrag.ask import init_pipeline
pipeline = init_pipeline()    # set load_vlm=False to skip Qwen if debugging retrieval
print("VLM loaded:", pipeline.vlm.loaded_name if pipeline.vlm else "none")
print("KG :", pipeline.kg.g.number_of_nodes(), "nodes,", pipeline.kg.g.number_of_edges(), "edges")

## 3. Ask

Try whatever you want. The answer is structured as MUTCD itself is:
**Standards** (mandatory) → **Guidance** (recommended) → **Options** (permitted) → **Visual evidence** → **Citations**.

In [ ]:
from mrag.ask import ask
_ = ask("What is required when installing a STOP sign at an all-way stop intersection?")

In [ ]:
_ = ask("Explain Figure 2B-1 and the plaques it shows", show_scores=True)

In [ ]:
_ = ask("What does MUTCD say about pedestrian hybrid beacons?", show_text=True)

## 4. Inspect the knowledge graph

The graph is your debugging window. Walk it directly when you want to understand why something was (or wasn't) retrieved.

In [ ]:
kg = pipeline.kg
g = kg.g

# Aggregate node counts by kind
from collections import Counter
kinds = Counter(d.get("kind") for _, d in g.nodes(data=True))
for k, n in sorted(kinds.items(), key=lambda kv: -kv[1]):
    print(f"  {k or '(?)':<10s} {n:>6d}")
print("\nedges by label:")
elabels = Counter(d.get("label") for *_, d in g.edges(data=True))
for k, n in sorted(elabels.items(), key=lambda kv: -kv[1]):
    print(f"  {k or '(?)':<18s} {n:>6d}")

In [ ]:
# Neighbourhood of a sign code
node = kg.sign("R1-3P")
print("sign node:", node)
print("data     :", g.nodes[node])
print("\n1-hop neighbours:")
for nb in sorted(kg.neighbors(node, n_hops=1)):
    if nb == node: continue
    print(f"  {nb}  ({g.nodes[nb].get('kind', '?')})")

In [ ]:
# Which figures does Section 2B.04 directly link to via cross-references in its chunks?
for ch_id in [d.get("id") for n, d in g.nodes(data=True)
              if d.get("kind") == "Chunk" and d.get("section") == "2B.04"]:
    figs = kg.figures_for_chunk(ch_id)
    if figs:
        print(f"  {ch_id}  -> {figs}")

## 5. Debug retrieval (no VLM call)

Useful when you want to see *what was retrieved* without spending 5 s on generation.

In [ ]:
from mrag.retrieval import Retriever
r = pipeline.retriever
for q in [
    "What's the difference between Standard and Guidance for stop signs?",
    "R1-3P all-way plaque",
    "pedestrian hybrid beacon signal sequence",
    "Figure 2B-1",
]:
    res = r.retrieve(q)
    print("=" * 92)
    print("Q:", q)
    print("-- top chunks --")
    for c in res.chunks[:5]:
        print(f"  Sec {c['section_id']:>8s} {c['content_type']:8s} §{c['ordinal']:>3d} "
              f"p{c['page_printed']:>4s}  s={c.get('score',0):.3f}  | {c['section_title'][:60]}")
    print("-- top figures --")
    for f in res.figures[:4]:
        print(f"  {f.get('figure_id','?'):<22s} p{f.get('page_printed','?'):>4s} | {(f.get('caption','') or '')[:80]}")
    print()